<img src="logos/Logo_de_la_Facultad_Experimental_de_Ciencia_y_Tecnología.svg.png" width="250" style="display: block; margin-left: auto; margin-right: auto;">

# Titanic – Preparación de Datos e Ingeniería de Características

## Propósito

Este notebook convierte los principales hallazgos del EDA en un flujo de preprocesamiento claro y reproducible. La prioridad es comprender **qué hace cada paso y por qué es necesario**.

Al finalizar podremos:

- validar el contrato mínimo de los datos de entrada;
- crear particiones de train y test sin ajustar previamente ninguna transformación;
- crear un conjunto pequeño de características justificadas por el EDA;
- imputar, codificar y escalar diferentes tipos de variables;
- reunir todas las transformaciones en un pipeline de scikit-learn;
- preparar la entrega al notebook de comparación de modelos sin consultar el conjunto de test.

```text
Datos crudos
    ↓
Train / test
    ↓
Ingeniería de características
    ↓
Imputación + codificación + escalado
    ↓
Modelo
```

> Un notebook complementario avanzado contiene transformers personalizados, imputación condicional de edad, características basadas en grupos de boletos, pipelines configurables y experimentos de ablación. El presente notebook enseña primero el camino esencial.

> **Limitación del conjunto de test:** el EDA anterior examinó el conjunto completo de datos etiquetados. Por tanto, el conjunto de test es útil para el flujo de la asignatura, pero no constituye un test final perfectamente intacto. En un proyecto nuevo, el conjunto de test final debe crearse y bloquearse antes del EDA detallado.

## 1. Importaciones y rutas del proyecto

Las dependencias deben instalarse desde el entorno del proyecto, no desde una celda con `pip install` dentro del notebook.

In [1]:
from __future__ import annotations

import json
import platform
import re
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.cluster import DBSCAN
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.utils.validation import check_is_fitted

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET_COLUMN = "Survived"

# Permitir la ejecución desde la raíz del proyecto o desde notebooks/.
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data" / "titanic_data.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "titanic_data.csv"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"No se encontró el dataset en {DATA_PATH}")

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

Python: 3.12.3
pandas: 3.0.1
scikit-learn: 1.8.0


## 2. Cargar y validar los datos crudos

El EDA identificó las columnas esperadas y los rangos válidos. Ahora convertimos esas observaciones en verificaciones ejecutables.

La validación se escribe deliberadamente mediante pasos explícitos. Si cambia el esquema de entrada, el notebook debe detenerse con un mensaje informativo en lugar de ignorar silenciosamente el problema.

In [2]:
required_columns = [
    "PassengerId",
    "Survived",
    "Pclass",
    "Name",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Ticket",
    "Fare",
    "Cabin",
    "Embarked",
]

df = pd.read_csv(DATA_PATH)

missing_columns = []
for column in required_columns:
    if column not in df.columns:
        missing_columns.append(column)

if missing_columns:
    raise ValueError(f"Faltan columnas requeridas: {missing_columns}")

if df["PassengerId"].duplicated().any():
    raise ValueError("PassengerId debe ser único.")

if not df["Pclass"].isin([1, 2, 3]).all():
    raise ValueError("Pclass solo puede contener 1, 2 o 3.")

if df["Age"].dropna().lt(0).any():
    raise ValueError("Age no puede ser negativa.")

if df["Fare"].dropna().lt(0).any():
    raise ValueError("Fare no puede ser negativa.")

if not df[TARGET_COLUMN].isin([0, 1]).all():
    raise ValueError("Survived solo puede contener 0 o 1.")

print(f"Dataset validado: {df.shape[0]} filas y {df.shape[1]} columnas")
df.head()

Dataset validado: 891 filas y 12 columnas


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 3. Separar la variable objetivo y crear las particiones

Realizamos la partición antes de ajustar imputadores, codificadores, escaladores o modelos. `stratify=y` conserva aproximadamente la misma proporción de la variable objetivo en ambas particiones.

Solo los datos de entrenamiento pueden utilizarse para validación cruzada, selección de características, comparación de modelos, Optuna o selección del umbral. Deliberadamente no mostramos la distribución de la variable objetivo del conjunto de test.

In [3]:
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Filas de entrenamiento: {len(X_train)}")
print(f"Filas de test: {len(X_test)}")

training_target_summary = (
    y_train
    .value_counts(normalize=True)
    .sort_index()
    .rename("proportion")
    .to_frame()
)
training_target_summary

Filas de entrenamiento: 712
Filas de test: 179


,proportion
Survived,
0,0.616573
1,0.383427


## 4. Traducir los hallazgos del EDA en decisiones de preparación

Este notebook implementa una única línea base fácil de comprender. Las representaciones alternativas permanecen como hipótesis para el notebook de modelado.

| Hallazgo del EDA | Decisión en esta línea base | Alternativa para comparar después |
|---|---|---|
| `Age` presenta ausencia estructurada | Imputación por mediana + indicador de ausencia | Imputación condicional por clase/título |
| `Fare` está fuertemente sesgada a la derecha | Usar `log1p(Fare)` | Conservar la tarifa original |
| `Pclass` puede no tener un efecto lineal | Codificación one-hot | Tratarla como ordinal para árboles |
| La estructura familiar es no lineal | Usar tamaño familiar e indicador de viaje en solitario | Conservar `SibSp` y `Parch` |
| La disponibilidad de cabina puede aportar información | Usar un indicador de cabina disponible | Usar la cubierta agrupada |
| El nombre contiene información social | Extraer y agrupar el título del pasajero | Excluir el título |

La línea base evita conservar simultáneamente representaciones exactamente redundantes.

## 5. Crear características comprensibles y configurables

La función siguiente crea una base común y, cuando la variante lo solicita, seis features de dominio. Ninguna aprende estadísticas entre filas. Esta separación permite ejecutar una ablación reproducible en el notebook 03.

Las features opcionales son `SexPclass`, `FamilyCategory`, `IsChild`, `IsMother`, `FarePerPersonLog1p` y `CabinDeck`.


In [4]:
def create_titanic_features(
    data: pd.DataFrame,
    include_domain_features: bool = False,
) -> pd.DataFrame:
    """Crear features deterministas; las opcionales se controlan por variante."""
    result = data.copy()

    title_pattern = r",\s*([^.]+)\."
    result["Title"] = result["Name"].str.extract(
        title_pattern,
        expand=False,
    )
    result["Title"] = result["Title"].replace({
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs",
    }).fillna("Unknown")

    result["FamilySize"] = result["SibSp"] + result["Parch"] + 1
    result["IsAlone"] = result["FamilySize"].eq(1).astype(int)
    result["HasCabin"] = result["Cabin"].notna().astype(int)
    result["AgeMissing"] = result["Age"].isna().astype(int)
    result["FareLog1p"] = np.log1p(result["Fare"])

    if include_domain_features:
        result["SexPclass"] = (
            result["Sex"].astype(str)
            + "_class_"
            + result["Pclass"].astype(str)
        )
        result["FamilyCategory"] = pd.cut(
            result["FamilySize"],
            bins=[0, 1, 4, 6, np.inf],
            labels=["Alone", "Small", "Medium", "Large"],
            include_lowest=True,
        ).astype(str)
        result["IsChild"] = (
            result["Age"].notna() & result["Age"].lt(14)
        ).astype(int)
        result["IsMother"] = (
            result["Sex"].eq("female")
            & result["Age"].ge(18)
            & result["Parch"].gt(0)
            & ~result["Title"].eq("Miss")
        ).astype(int)
        fare_per_person = result["Fare"] / result["FamilySize"]
        result["FarePerPersonLog1p"] = np.log1p(fare_per_person)
        result["CabinDeck"] = result["Cabin"].str[0].fillna("Unknown")

    return result


UNSUPERVISED_INPUT_FEATURES = (
    "Age",
    "FareLog1p",
    "FamilySize",
    "Pclass",
)


class UnsupervisedFeatureTransformer(BaseEstimator, TransformerMixin):
    """Crear features de DBSCAN e Isolation Forest sin utilizar el target."""

    def __init__(
        self,
        dbscan_eps: float = 0.9,
        dbscan_min_samples: int = 10,
        contamination: float = 0.05,
        random_state: int = 42,
    ):
        self.dbscan_eps = dbscan_eps
        self.dbscan_min_samples = dbscan_min_samples
        self.contamination = contamination
        self.random_state = random_state

    def fit(self, X: pd.DataFrame, y=None):
        """Aprender imputación, escalado, clústeres y anomalías solo con X."""
        self.input_pipeline_ = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]
        )
        matrix = self.input_pipeline_.fit_transform(
            X.loc[:, UNSUPERVISED_INPUT_FEATURES]
        )

        self.dbscan_ = DBSCAN(
            eps=self.dbscan_eps,
            min_samples=self.dbscan_min_samples,
        ).fit(matrix)

        self.isolation_forest_ = IsolationForest(
            n_estimators=200,
            contamination=self.contamination,
            random_state=self.random_state,
            n_jobs=1,
        ).fit(matrix)

        self.training_cluster_labels_ = self.dbscan_.labels_.copy()
        self.training_anomaly_scores_ = (
            self.isolation_forest_.decision_function(matrix)
        )
        self.training_outlier_flags_ = (
            self.training_anomaly_scores_ < 0
        ).astype(int)

        core_indices = self.dbscan_.core_sample_indices_
        self.core_cluster_labels_ = self.dbscan_.labels_[core_indices]

        if len(core_indices) > 0:
            self.core_neighbor_model_ = NearestNeighbors(
                n_neighbors=1
            ).fit(self.dbscan_.components_)
        else:
            self.core_neighbor_model_ = None

        return self

    def _cluster_new_rows(self, matrix: np.ndarray) -> np.ndarray:
        """Asignar el clúster del punto núcleo más cercano dentro de eps."""
        labels = np.full(matrix.shape[0], -1, dtype=int)

        if self.core_neighbor_model_ is None:
            return labels

        distances, neighbor_indices = self.core_neighbor_model_.kneighbors(
            matrix
        )
        inside_eps = distances[:, 0] <= self.dbscan_eps
        labels[inside_eps] = self.core_cluster_labels_[
            neighbor_indices[inside_eps, 0]
        ]
        return labels

    @staticmethod
    def _append_features(
        X: pd.DataFrame,
        cluster_labels: np.ndarray,
        anomaly_scores: np.ndarray,
    ) -> pd.DataFrame:
        result = X.copy()
        result["cluster"] = cluster_labels.astype(str)
        result["anomalyScore"] = anomaly_scores.astype(float)
        result["isOutlier"] = (anomaly_scores < 0).astype(int)
        return result

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Aplicar los objetos aprendidos durante fit a observaciones nuevas."""
        check_is_fitted(
            self,
            ["input_pipeline_", "dbscan_", "isolation_forest_"],
        )
        matrix = self.input_pipeline_.transform(
            X.loc[:, UNSUPERVISED_INPUT_FEATURES]
        )
        cluster_labels = self._cluster_new_rows(matrix)
        anomaly_scores = self.isolation_forest_.decision_function(matrix)
        return self._append_features(X, cluster_labels, anomaly_scores)

    def fit_transform(self, X: pd.DataFrame, y=None, **fit_params) -> pd.DataFrame:
        """Conservar las etiquetas nativas de DBSCAN sobre sus datos de ajuste."""
        self.fit(X, y)
        return self._append_features(
            X,
            self.training_cluster_labels_,
            self.training_anomaly_scores_,
        )


### Inspeccionar el resultado antes de construir el pipeline

Durante el desarrollo conviene inspeccionar algunas filas para verificar que cada característica nueva representa lo que pretendíamos. Esta vista previa tiene fines docentes y de depuración; la transformación final continuará ocurriendo dentro del pipeline.

In [5]:
preview = create_titanic_features(
    X_train.head(),
    include_domain_features=True,
)

columns_to_preview = [
    "FamilySize", "IsAlone", "HasCabin", "AgeMissing", "FareLog1p", "Title",
    "SexPclass", "FamilyCategory", "IsChild", "IsMother",
    "FarePerPersonLog1p", "CabinDeck",
]

preview[columns_to_preview]


,FamilySize,IsAlone,HasCabin,AgeMissing,FareLog1p,Title,SexPclass,FamilyCategory,IsChild,IsMother,FarePerPersonLog1p,CabinDeck
692,1,1,0,1,4.051712,Mr,male_class_3,Alone,0,0,4.051712,Unknown
481,1,1,0,1,0.000000,Mr,male_class_2,Alone,0,0,0.000000,Unknown
527,1,1,1,1,5.406181,Mr,male_class_1,Alone,0,0,5.406181,C
855,2,0,0,0,2.336987,Mrs,female_class_3,Small,0,1,1.736071,Unknown
801,3,0,0,0,3.305054,Mrs,female_class_2,Small,0,1,2.277267,Unknown


## 6. Definir las variantes de features

Se compararán cinco variantes. Todas parten de las mismas features base. Las técnicas no supervisadas permanecen dentro del pipeline; por tanto, se ajustarán nuevamente en cada fold.

| Variante | Dominio | DBSCAN + `isOutlier` | `anomalyScore` |
|---|---:|---:|---:|
| `baseline` | No | No | No |
| `domain_only` | Sí | No | No |
| `unsupervised_binary` | No | Sí | No |
| `unsupervised_with_score` | No | Sí | Sí |
| `domain_and_unsupervised` | Sí | Sí | No |


In [6]:
BASE_NUMERIC_FEATURES = ["Age", "FareLog1p", "FamilySize"]
BASE_CATEGORICAL_FEATURES = ["Sex", "Embarked", "Pclass", "Title"]
BASE_BINARY_FEATURES = ["AgeMissing", "IsAlone", "HasCabin"]

DOMAIN_NUMERIC_FEATURES = ["FarePerPersonLog1p"]
DOMAIN_CATEGORICAL_FEATURES = ["SexPclass", "FamilyCategory", "CabinDeck"]
DOMAIN_BINARY_FEATURES = ["IsChild", "IsMother"]

FEATURE_VARIANTS = {
    "baseline": {
        "domain": False,
        "unsupervised": False,
        "anomaly_score": False,
    },
    "domain_only": {
        "domain": True,
        "unsupervised": False,
        "anomaly_score": False,
    },
    "unsupervised_binary": {
        "domain": False,
        "unsupervised": True,
        "anomaly_score": False,
    },
    "unsupervised_with_score": {
        "domain": False,
        "unsupervised": True,
        "anomaly_score": True,
    },
    "domain_and_unsupervised": {
        "domain": True,
        "unsupervised": True,
        "anomaly_score": False,
    },
}


## 7. Construir el preprocesador correspondiente a cada variante

La función arma listas nuevas para cada experimento. Las variables numéricas se imputan y escalan; las categóricas se imputan y codifican; los indicadores binarios pasan sin cambios.


In [7]:
def build_preprocessor(feature_variant: str) -> ColumnTransformer:
    """Construir las columnas de entrada correspondientes a una variante."""
    config = FEATURE_VARIANTS[feature_variant]

    numeric_features = BASE_NUMERIC_FEATURES.copy()
    categorical_features = BASE_CATEGORICAL_FEATURES.copy()
    binary_features = BASE_BINARY_FEATURES.copy()

    if config["domain"]:
        numeric_features += DOMAIN_NUMERIC_FEATURES
        categorical_features += DOMAIN_CATEGORICAL_FEATURES
        binary_features += DOMAIN_BINARY_FEATURES

    if config["unsupervised"]:
        categorical_features += ["cluster"]
        binary_features += ["isOutlier"]

    if config["anomaly_score"]:
        numeric_features += ["anomalyScore"]

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore", min_frequency=0.01),
        ),
    ])

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
            ("binary", "passthrough", binary_features),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


## 8. Inspeccionar una variante completa

La variante más amplia se utiliza únicamente para verificar la estructura. Esto no implica que vaya a ser seleccionada: la decisión se tomará por validación cruzada en el notebook 03.


In [8]:
demonstration_variant = "domain_and_unsupervised"
preprocessor = build_preprocessor(demonstration_variant)
preprocessor


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``featu

## 9. Colocar dentro del pipeline

El pipeline siguiente recibe una variante. DBSCAN e Isolation Forest solo se incluyen cuando corresponde y se ajustan exclusivamente durante `fit`.


In [9]:
def build_preparation_pipeline(feature_variant: str) -> Pipeline:
    config = FEATURE_VARIANTS[feature_variant]
    steps = [
        (
            "feature_engineering",
            FunctionTransformer(
                create_titanic_features,
                kw_args={"include_domain_features": config["domain"]},
                validate=False,
            ),
        ),
    ]

    if config["unsupervised"]:
        steps.append((
            "unsupervised_features",
            UnsupervisedFeatureTransformer(random_state=RANDOM_STATE),
        ))

    steps.append(("preprocessor", build_preprocessor(feature_variant)))
    return Pipeline(steps)


preparation_pipeline = build_preparation_pipeline(demonstration_variant)
X_train_prepared = preparation_pipeline.fit_transform(X_train, y_train)
output_feature_names = (
    preparation_pipeline.named_steps["preprocessor"].get_feature_names_out()
)

unsupervised_step = preparation_pipeline.named_steps["unsupervised_features"]
cluster_counts = pd.Series(
    unsupervised_step.training_cluster_labels_, name="cluster"
).value_counts().sort_index()
outlier_rate = unsupervised_step.training_outlier_flags_.mean()

print(f"Variante demostrada: {demonstration_variant}")
print(f"Dimensiones crudas: {X_train.shape}")
print(f"Dimensiones preparadas: {X_train_prepared.shape}")
print("Distribución de clústeres:")
print(cluster_counts)
print(f"Proporción de anomalías: {outlier_rate:.1%}")
pd.DataFrame({"caracteristica_salida": output_feature_names})


Variante demostrada: domain_and_unsupervised
Dimensiones crudas: (712, 11)
Dimensiones preparadas: (712, 45)
Distribución de clústeres:
cluster
-1     44
 0    159
 1    372
 2    137
Name: count, dtype: int64
Proporción de anomalías: 5.1%


,caracteristica_salida
0,Age
1,FareLog1p
2,FamilySize
3,FarePerPersonLog1p
4,Sex_female
5,Sex_male
6,Embarked_C
7,Embarked_Q
8,Embarked_S
9,Pclass_1


### Verificaciones estructurales

Estas comprobaciones aseguran que la variante de demostración conserva todas las filas, produce valores finitos e incluye las features solicitadas.


In [10]:
assert X_train_prepared.shape[0] == X_train.shape[0]
assert X_train_prepared.shape[1] == len(output_feature_names)

values_to_check = (
    X_train_prepared.toarray()
    if hasattr(X_train_prepared, "toarray")
    else X_train_prepared
)
assert np.isfinite(values_to_check).all()
assert "isOutlier" in output_feature_names
assert "FarePerPersonLog1p" in output_feature_names
assert any(name.startswith("SexPclass_") for name in output_feature_names)
assert any(name.startswith("FamilyCategory_") for name in output_feature_names)
assert any(name.startswith("CabinDeck_") for name in output_feature_names)
print("Todas las verificaciones estructurales fueron superadas.")


Todas las verificaciones estructurales fueron superadas.


## 10. Construir el futuro pipeline desde los datos crudos hasta el modelo

El notebook 03 proporcionará tanto un estimador como una variante. `clone` evita compartir estado aprendido entre experimentos.


In [11]:
def build_model_pipeline(estimator, feature_variant: str) -> Pipeline:
    preparation_steps = build_preparation_pipeline(feature_variant).steps
    return Pipeline(
        steps=preparation_steps + [("model", clone(estimator))]
    )

# Ejemplo:
# build_model_pipeline(LogisticRegression(), "baseline")


## 11. Guardar un manifiesto legible de la partición

Guardamos la asignación de cada pasajero a una partición para que otro notebook pueda reproducir exactamente el split. La construcción es intencionalmente explícita: crear cada tabla, asignarle su etiqueta, combinar ambas tablas y ordenar el resultado.

**No** guardamos la matriz ya transformada. El pipeline final del modelo debe aprender su preprocesamiento dentro de la validación cruzada.

In [12]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

train_manifest = X_train[["PassengerId"]].copy()
train_manifest["split"] = "train"

test_manifest = X_test[["PassengerId"]].copy()
test_manifest["split"] = "test"

split_manifest = pd.concat(
    [train_manifest, test_manifest],
    ignore_index=True,
)

split_manifest = split_manifest.sort_values("PassengerId")

split_manifest.to_csv(
    ARTIFACTS_DIR / "split_assignments_teaching_es.csv",
    index=False,
)

split_manifest.head()

,PassengerId,split
607,1,train
209,2,train
142,3,train
633,4,train
810,5,test


## 12. Guardar metadatos mínimos de reproducibilidad

En esta etapa todavía no existe un modelo ajustado y seleccionado. Por ello no serializamos un pipeline de producción. Guardamos la información necesaria para reproducir la partición, el entorno y las features no supervisadas.

In [13]:
metadata = {
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "target_column": TARGET_COLUMN,
    "unsupervised_features": {
        "input_columns": list(UNSUPERVISED_INPUT_FEATURES),
        "dbscan_eps": unsupervised_step.dbscan_eps,
        "dbscan_min_samples": unsupervised_step.dbscan_min_samples,
        "isolation_forest_contamination": unsupervised_step.contamination,
        "anomaly_score_method": "decision_function",
        "is_outlier_threshold": 0.0,
    },
    "feature_variants": FEATURE_VARIANTS,
    "domain_features": [
        "SexPclass", "FamilyCategory", "IsChild",
        "IsMother", "FarePerPersonLog1p", "CabinDeck",
    ],
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__,
    "test_note": (
        "El EDA anterior examinó el conjunto completo de datos etiquetados. "
        "Debe tratarse como conjunto de test, no como un test final perfectamente intacto."
    ),
}

metadata_path = ARTIFACTS_DIR / "data_preparation_teaching_metadata_es.json"

with metadata_path.open("w", encoding="utf-8") as metadata_file:
    json.dump(metadata, metadata_file, indent=2)

print(f"Metadatos guardados en: {metadata_path}")

Metadatos guardados en: /mnt/c/Users/aespinoza/PycharmProjects/titanic_ml_solution/artifacts/data_preparation_teaching_metadata_es.json


## 13. Entrega al notebook de modelado

El notebook 03 debe:

1. evaluar todas las variantes con los mismos folds y modelos iniciales;
2. seleccionar features exclusivamente con validación cruzada sobre train;
3. ajustar hiperparámetros después de seleccionar la variante;
4. evaluar test una sola vez al final de la ejecución;
5. evitar resultados históricos escritos manualmente.

### Comprueba tu comprensión

- ¿Por qué DBSCAN e Isolation Forest deben estar dentro del pipeline?
- ¿Por qué `anomalyScore` constituye una variante diferente de `isOutlier`?
- ¿Por qué las mismas particiones deben utilizarse para comparar las variantes?
- ¿Por qué no debe elegirse una variante observando test?
